In [14]:
import os
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()
client = Anthropic()

def ask(prompt: str, system: str = "", model: str = "claude-haiku-4-5-20251001") -> str:
    kwargs = {"model": model, "max_tokens": 512,
              "messages": [{"role": "user", "content": prompt}]}
    if system:
        kwargs["system"] = system
    r = client.messages.create(**kwargs)
    return r.content[0].text

# Classification
# result = ask(
#     "Classify this support ticket into one of: billing, technical, account, other.\n\n"
#     "Ticket: I can't log into my account after resetting my password."
# )

# Extraction
# result = ask(
#     "Extract the name, email, and years of experience from this text:\n\n"
#     "Hi, I'm Shivam Prajapati. You can reach me at shivam@email.com. "
#     "I have 7 years of experience in backend development."
# )

# Tranformation
# result = ask(
#     "Rewrite this Slack message as a formal email:\n\n"
#     "yo the deploy broke prod again, can someone take a look asap??"
# )

# Generation with constaints
# result = ask(
#     "Write a git commit message for this change:\n\n"
#     "Added retry logic to the payment API call — it now retries up to 3 times "
#     "with exponential backoff before failing."
# )

# Summary with format
# result = ask(
#     "Summarise this in 3 bullet points:\n\n"
#     "Retrieval-Augmented Generation (RAG) is a technique that combines a "
#     "large language model with an external knowledge base. Instead of relying "
#     "solely on its training data, the model retrieves relevant documents at "
#     "inference time and uses them to ground its response. This reduces "
#     "hallucinations and allows the model to answer questions about private or "
#     "recent data it was never trained on."
# )
# print(result)

# result = ask("""Classify support tickets into: billing, technical, account, other.
# Reply with the category name only — nothing else.

# Examples:
# Ticket: My invoice shows the wrong amount.
# Category: billing

# Ticket: The app crashes when I upload a file over 10MB.
# Category: technical

# Ticket: I want to change my username but the option is greyed out.
# Category: account

# Now classify:
# Ticket: I can't log into my account after resetting my password.
# Category:""")
# print(repr(result))

# TASK 4 fixed: Commit messages with few-shot conventional commits format
# Problem: zero-shot doesn't know you want the feat/fix/chore prefix
# Fix: show the exact format with 3 examples

# result = ask("""Write git commit messages using conventional commits format.
# Format: type(scope): short description

# Examples:
# Change: Added email validation to the registration form
# Commit: feat(auth): add email validation on registration

# Change: Fixed a null pointer crash in the cart checkout flow
# Commit: fix(checkout): resolve null pointer exception on empty cart

# Change: Updated README with local development setup instructions
# Commit: docs(readme): add local development setup guide

# Now write:
# Change: Added retry logic to the payment API call — retries 3 times with exponential backoff
# Commit:""")
# print(result)


# THE TASK: Analyse a resume bullet point and suggest improvements
# We'll run 5 increasingly better prompts for the exact same task

bullet = "Worked on APIs and improved performance of the system"

# PROMPT VERSION 1 — bare minimum
v1 = ask(f"Improve this resume bullet: {bullet}")

# PROMPT VERSION 2 — add context
v2 = ask(f"Improve this resume bullet for a software engineer job application: {bullet}")

# PROMPT VERSION 3 — add format constraint
v3 = ask(
    f"Improve this resume bullet for a software engineer job application. "
    f"Return ONLY the improved bullet, nothing else.\n\n{bullet}"
)

# PROMPT VERSION 4 — add persona + constraints
v4 = ask(
    prompt=f"Improve this resume bullet. Return ONLY the improved version.\n\n{bullet}",
    system="You are a senior technical recruiter at a top tech company. "
           "You rewrite resume bullets to be specific, quantified, and action-verb led. "
           "Never use vague words like 'worked on', 'helped', or 'involved in'."
)

# PROMPT VERSION 5 — few-shot examples + persona
v5 = ask(
    prompt="""Improve this resume bullet. Return ONLY the improved version.

Examples:
Weak: Helped with the frontend redesign
Strong: Engineered React component library used by 8-person team, cutting UI development time by 40%

Weak: Worked on database queries
Strong: Optimised 12 PostgreSQL queries using indexing and query restructuring, reducing p95 latency from 800ms to 95ms

Weak: Did some work on the API
Strong: Designed and shipped 6 RESTful API endpoints serving 50K daily requests with 99.9% uptime

Now improve:
Weak: """ + bullet + """
Strong:""",
    system="You are a senior technical recruiter. Be specific and quantified. One line only."
)

# Print all 5 side by side
for i, (label, result) in enumerate([
    ("V1 bare", v1), ("V2 context", v2), ("V3 format", v3),
    ("V4 persona", v4), ("V5 few-shot", v5)
], 1):
    print(f"\n{'='*50}")
    print(f"Prompt {i} ({label}):")
    print(result.strip())


Prompt 1 (V1 bare):
# Improved Resume Bullets

Here are several stronger versions, depending on your context:

## If you optimized performance:
- **Optimized API response times by 40%, reducing system latency from 800ms to 480ms and improving user experience for 50K+ daily active users**

## If you built/designed APIs:
- **Designed and implemented 12 RESTful APIs serving real-time data to mobile and web clients, handling 10M+ requests daily**

## If you debugged/maintained APIs:
- **Refactored legacy API endpoints, reducing database query time by 35% and decreasing server load by 25%**

## General best practices applied:
✓ **Specific metrics** (percentages, numbers, time reductions)
✓ **Quantified impact** (users affected, requests handled, revenue impact)
✓ **Clear action verbs** (Optimized, Designed, Refactored vs. "Worked on")
✓ **Business value** (what the improvement enabled)

**Which context best matches your work?** I can refine further with details like:
- What type of APIs? (

In [16]:
puzzle = """A store sells apples for $1.20, bananas for $0.50, oranges for $0.80.
Sam buys 4 apples, 6 bananas, and 2 oranges. He pays with a $20 bill.
How much change does Sam receive?"""

no_cot = ask(puzzle)
print("Without CoT:", no_cot)

# Now WITH chain-of-thought
with_cot = ask(puzzle + "\n\nThink step by step before giving the final answer.")
print("\nWith CoT:", with_cot)

Without CoT: # Calculating Sam's Change

**Cost of each item:**
- Apples: 4 × $1.20 = $4.80
- Bananas: 6 × $0.50 = $3.00
- Oranges: 2 × $0.80 = $1.60

**Total cost:**
$4.80 + $3.00 + $1.60 = $9.40

**Change received:**
$20.00 - $9.40 = **$10.60**

With CoT: # Step-by-step calculation

**Step 1: Calculate the cost of apples**
- 4 apples × $1.20 = $4.80

**Step 2: Calculate the cost of bananas**
- 6 bananas × $0.50 = $3.00

**Step 3: Calculate the cost of oranges**
- 2 oranges × $0.80 = $1.60

**Step 4: Calculate total cost**
- $4.80 + $3.00 + $1.60 = $9.40

**Step 5: Calculate change**
- $20.00 - $9.40 = $10.60

**Sam receives $10.60 in change.**


In [23]:
# START: weak version - without system prompt"
test_input = "Review this code: result = [x for x in range(1000000) if x % 2 == 0]"

weak  = ask(test_input)
# print("WEAK (no system):\n", weak)

#ADD role only
with_role = ask(test_input,system="You are a senior python engineer")
# print(with_role)

# ADD ROLE + CONTEXT
with_context = ask(test_input,
    system="You are a senior Python engineer reviewing code written by a junior developer "
           "who is learning best practices for the first time.")
# print(with_context)

# ADD ROLE + CONTEXT + CONSTRAINTS
with_constraints = ask(test_input,
    system="You are a senior Python engineer reviewing code for a junior developer.\n"
           "Constraints:\n"
           "- Focus only on performance and memory issues\n"
           "- Max 3 points\n"
           "- Never rewrite the entire snippet — show targeted changes only\n"
           "- Explain WHY each change matters, not just what to change")
#print(with_constraints)

# ADD ALL 4 PARTS — the complete formula
full_4part = ask(test_input,
    system="""ROLE:
You are a senior Python performance engineer with 10 years of experience
optimising production systems at scale.

CONTEXT:
The user is a junior developer learning Python best practices for the first time.
They understand basic syntax but haven't learned performance optimisation yet.

CONSTRAINTS:
- Focus only on performance and memory — not style or readability
- Maximum 3 points, ordered by severity
- Never rewrite the entire snippet — show a targeted fix for each point
- Explain WHY with a concrete example of when it would cause a real problem
- Do not use jargon without defining it

OUTPUT FORMAT:
For each issue:
**Issue [N]: [Short title]**
Problem: [one sentence]
Fix: [code snippet]
Why it matters: [one concrete scenario where this causes a real production issue]""")

# Compare all 4 versions
for label, result in [
    ("No system", weak),
    ("Role only", with_role),
    ("Role+Context", with_context),
    ("Role+Context+Constraints", with_constraints),
    ("Full 4-part", full_4part)
]:
    print(f"\n{'='*50}")
    print(f"VERSION: {label}")
    print(result.strip())



VERSION: No system
# Code Review

## Issues

### 1. **Performance Problem (Major)**
This creates a list with **500,000 elements in memory** all at once. For large ranges, this is wasteful.

**Better approach:**
```python
result = (x for x in range(1000000) if x % 2 == 0)  # Generator
```

Or if you need a list:
```python
result = list(range(0, 1000000, 2))  # More efficient
```

### 2. **Inefficient Even Number Check**
`x % 2 == 0` works but `x % 2` is more Pythonic for simple parity checks.

### 3. **Unclear Intent**
The code's purpose isn't immediately obvious.

## Improved Version

```python
# If you need a generator (memory efficient):
even_numbers = (x for x in range(0, 1000000, 2))

# If you need a list:
even_numbers = list(range(0, 1000000, 2))

# If you want filtering logic that's clearer:
even_numbers = filter(lambda x: x % 2 == 0, range(1000000))
```

## Benchmark
```python
# Your way: ~40MB memory
# range(0, 1000000, 2): ~28 bytes (just the range object)
# Generator: ~200 b

In [ ]:
plain = ask(
    "Review this API design: POST /users creates a user, GET /user gets all users",
    system="You are a senior API architect. Review for REST conventions. "
           "Point out naming issues. Max 3 issues. Format as numbered list. "
           "Be concise. Focus on conventions not implementation."
)

xml_prompt = ask(
    "Review this API design: POST /users creates a user, GET /user gets all users",
    system="""<role>Senior REST API architect with 10 years of API design experience</role>

<task>Review the API design for REST convention violations and naming issues.</task>

<rules>
- Max 3 issues, ordered by severity
- Focus only on REST conventions — not implementation or performance
- Reference the specific HTTP method and path for each issue
</rules>

<format>
1. [Issue title] — [one sentence explanation]
   Fix: [corrected endpoint]
</format>"""
)

print("PLAIN:\n", plain)
print("\nXML:\n", xml_prompt)

PLAIN:
 # REST Convention Issues

1. **Inconsistent singular/plural naming** – Use `GET /users` (plural) to list all users, not `GET /user`. Singular endpoints should represent individual resources (e.g., `GET /users/{id}`).

2. **POST endpoint naming** – `POST /users` is correct for creation, but ensure it matches your retrieval pattern. Currently `POST /users` and `GET /user` are inconsistent.

3. **Missing resource identifier** – To get a specific user, you need `GET /users/{id}`. The current design lacks a way to retrieve individual resources by ID.

XML:
 # REST API Design Review

1. **Inconsistent Resource Naming** — The creation endpoint uses `/users` (plural) while the retrieval endpoint uses `/user` (singular), violating REST convention that the same resource should have a consistent path.
   Fix: `GET /users` to retrieve all users

2. **Incorrect HTTP Method for Collection Retrieval** — Using POST to create a single resource is correct, but the GET endpoint should retrieve a 